# BỘ KIỂM THỬ TẢI PHASE 1 TRÊN KAGGLE CPU (3 KỊCH BẢN RIÊNG BIỆT)
Gồm 3 kịch bản đo đạc chuyên sâu các khía cạnh kiến trúc khác nhau:
1. **Kịch bản 1: Baseline Read Capacity (50 VUs)** — Đo thông lượng đọc cơ bản.
2. **Kịch bản 2: Stress Testing (100 -> 300 VUs)** — Ép tải tìm điểm gãy (Breaking point).
3. **Kịch bản 3: Auth & Write Lifecycle (30 VUs)** — Đo chi phí CPU của BCrypt và tầng Bảo mật JWT.

### Bước 1: Cài đặt OpenJDK 21, MySQL Server và k6 (Non-interactive)

In [ ]:
%%bash
export DEBIAN_FRONTEND=noninteractive
export NEEDRESTART_MODE=a

apt-get update -qq
apt-get install -y -qq --no-install-recommends openjdk-21-jre-headless mysql-server curl

if [ ! -f /usr/local/bin/k6 ]; then
    curl -s https://github.com/grafana/k6/releases/download/v0.49.0/k6-v0.49.0-linux-amd64.tar.gz -L | tar xvz
    mv k6-v0.49.0-linux-amd64/k6 /usr/local/bin/
    rm -rf k6-v0.49.0-linux-amd64
fi

java -version
k6 version
echo "==> BƯỚC 1 HOÀN TẤT THÀNH CÔNG!"

### Bước 2: Khởi động MySQL và Khởi tạo Database

In [ ]:
%%bash
mkdir -p /var/run/mysqld
chown -R mysql:mysql /var/run/mysqld /var/lib/mysql
service mysql start

for i in {1..15}; do
    if mysqladmin ping --silent; then
        echo "==> MySQL daemon đã sẵn sàng!"
        break
    fi
    sleep 1
done

mysql -u root -e "CREATE DATABASE IF NOT EXISTS job_application_db;"
mysql -u root -e "CREATE USER IF NOT EXISTS 'job_db_user'@'localhost' IDENTIFIED BY 'job_db_password';"
mysql -u root -e "GRANT ALL PRIVILEGES ON job_application_db.* TO 'job_db_user'@'localhost';"
mysql -u root -e "FLUSH PRIVILEGES;"

echo "==> BƯỚC 2 HOÀN TẤT: Database job_application_db đã sẵn sàng!"

### Bước 3: Khởi chạy Spring Boot dưới nền (Python Subprocess)

In [ ]:
import subprocess, glob

jar_files = glob.glob('/kaggle/input/**/app.jar', recursive=True) + \
            glob.glob('/kaggle/input/**/*.jar', recursive=True) + \
            glob.glob('/kaggle/working/**/app.jar', recursive=True)
jar_files = [f for f in jar_files if not f.endswith('.original')]

if not jar_files:
    raise FileNotFoundError("CHƯA TÌM THẤY file app.jar! Hãy đảm bảo bạn đã Add Dataset chứa file JAR vào Notebook.")

jar_path = jar_files[0]
print(f"==> Đang dùng file: {jar_path}")

log_file = open("backend.log", "w")
process = subprocess.Popen(
    ["java", "-jar", jar_path],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)
print(f"==> BƯỚC 3 HOÀN TẤT: Backend đã chạy dưới nền! PID: {process.pid}")

### Bước 4: Kiểm tra trạng thái Backend và Nạp Seed Data

In [ ]:
%%bash
echo "==> Chờ 15s cho Spring Boot khởi động và Flyway chạy migration..."
sleep 15

echo "--- 15 dòng log mới nhất của Backend ---"
tail -n 15 backend.log

SEED_PATH=$(find /kaggle/input /kaggle/working -name "seed_data.sql" | head -n 1)
if [ -n "$SEED_PATH" ]; then
    echo "==> Đang nạp seed data từ: $SEED_PATH"
    mysql -u job_db_user -pjob_db_password job_application_db < "$SEED_PATH"
    echo "==> Đã nạp xong seed data!"
fi

echo "==> Gọi thử API GET /api/jobs:"
curl -s --connect-timeout 5 http://localhost:8080/api/jobs?page=0&size=2
echo ""

---
## 🧪 KỊCH BẢN 1: BASELINE READ CAPACITY (50 VUs - 40s)
Đo thông lượng đọc cơ bản của hệ thống Pha 1 (Xem danh sách, tìm kiếm, xem chi tiết).

In [ ]:
%%writefile /kaggle/working/test_scenario1_baseline_read.js
import http from 'k6/http';
import { check, sleep } from 'k6';

export const options = {
  stages: [
    { duration: '10s', target: 30 },
    { duration: '20s', target: 50 },
    { duration: '10s', target: 0 },
  ],
  thresholds: {
    http_req_failed: ['rate<0.05'],
    http_req_duration: ['p(95)<1500'],
  },
};

const BASE_URL = 'http://localhost:8080/api';

export default function () {
  const resList = http.get(`${BASE_URL}/jobs?page=0&size=10`);
  check(resList, { '1. GET /jobs 200': (r) => r.status === 200 });
  sleep(0.1);

  const resFilter = http.get(`${BASE_URL}/jobs?keyword=Java&category=IT`);
  check(resFilter, { '2. GET /jobs search 200': (r) => r.status === 200 });
  sleep(0.1);

  const randomJobId = Math.floor(Math.random() * 50) + 1;
  const resDetail = http.get(`${BASE_URL}/jobs/${randomJobId}`);
  check(resDetail, { '3. GET /jobs/{id} 200': (r) => r.status === 200 });
  sleep(0.2);
}

In [ ]:
%%bash
echo "==> CHẠY KỊCH BẢN 1: BASELINE READ (40s)..."
k6 run --summary-export=summary_scenario1.json /kaggle/working/test_scenario1_baseline_read.js

---
## 🧪 KỊCH BẢN 2: STRESS TEST - TÌM ĐIỂM GÃY (100 -> 300 VUs - 60s)
Ép tải cực hạn để quan sát thời điểm CPU 100%, MySQL cạn Connection Pool và xuất hiện lỗi.

In [ ]:
%%writefile /kaggle/working/test_scenario2_stress_test.js
import http from 'k6/http';
import { check, sleep } from 'k6';

export const options = {
  stages: [
    { duration: '10s', target: 100 },
    { duration: '20s', target: 200 },
    { duration: '20s', target: 300 },
    { duration: '10s', target: 0 },
  ],
  thresholds: {
    http_req_duration: ['p(95)<3000'],
  },
};

const BASE_URL = 'http://localhost:8080/api';

export default function () {
  const resList = http.get(`${BASE_URL}/jobs?page=0&size=20`);
  check(resList, { 'GET /jobs 200': (r) => r.status === 200 });

  const resFilter = http.get(`${BASE_URL}/jobs?keyword=Engineer&location=Hanoi`);
  check(resFilter, { 'GET /jobs search 200': (r) => r.status === 200 });

  const randomJobId = Math.floor(Math.random() * 50) + 1;
  const resDetail = http.get(`${BASE_URL}/jobs/${randomJobId}`);
  check(resDetail, { 'GET /jobs/{id} 200': (r) => r.status === 200 });

  sleep(0.05);
}

In [ ]:
%%bash
echo "==> CHẠY KỊCH BẢN 2: STRESS TEST (60s)..."
k6 run --summary-export=summary_scenario2.json /kaggle/working/test_scenario2_stress_test.js

---
## 🧪 KỊCH BẢN 3: AUTHENTICATION & WRITE OPERATIONS (30 VUs - 40s)
Đo đạc chi phí mã hóa BCrypt của CPU và khả năng xử lý nghiệp vụ Ghi + Xác thực JWT.

In [ ]:
%%writefile /kaggle/working/test_scenario3_auth_and_write.js
import http from 'k6/http';
import { check, sleep } from 'k6';

export const options = {
  stages: [
    { duration: '10s', target: 15 },
    { duration: '20s', target: 30 },
    { duration: '10s', target: 0 },
  ],
  thresholds: {
    http_req_failed: ['rate<0.05'],
    http_req_duration: ['p(95)<2500'],
  },
};

const BASE_URL = 'http://localhost:8080/api';

export default function () {
  const uniqueId = `${__VU}_${__ITER}_${Date.now()}`;
  const userEmail = `candidate_${uniqueId}@test.com`;
  const userPassword = 'Password123';

  // 1. Đăng ký tài khoản ứng viên (Write + BCrypt Hash)
  const registerPayload = JSON.stringify({
    email: userEmail,
    password: userPassword,
    fullName: `Candidate ${uniqueId}`,
    phone: '0912345678',
    location: 'Hanoi'
  });
  const headers = { 'Content-Type': 'application/json' };
  const regRes = http.post(`${BASE_URL}/auth/applicants/register`, registerPayload, { headers });
  check(regRes, { '1. POST /register 201': (r) => r.status === 201 });
  sleep(0.1);

  // 2. Đăng nhập lấy JWT (BCrypt Verify + JWT Sign)
  const loginPayload = JSON.stringify({
    email: userEmail,
    password: userPassword,
  });
  const loginRes = http.post(`${BASE_URL}/auth/applicants/login`, loginPayload, { headers });
  const loginSuccess = check(loginRes, { '2. POST /login 200': (r) => r.status === 200 });

  // 3. Đọc có bảo mật: Lấy profile bằng Bearer Token (JWT Verification Filter)
  if (loginSuccess && loginRes.json('accessToken')) {
    const token = loginRes.json('accessToken');
    const authHeaders = {
      'Content-Type': 'application/json',
      'Authorization': `Bearer ${token}`
    };
    const profileRes = http.get(`${BASE_URL}/applicants/me`, { headers: authHeaders });
    check(profileRes, { '3. GET /applicants/me 200': (r) => r.status === 200 });
  }
  sleep(0.2);
}

In [ ]:
%%bash
echo "==> CHẠY KỊCH BẢN 3: AUTH & WRITE (40s)..."
k6 run --summary-export=summary_scenario3.json /kaggle/working/test_scenario3_auth_and_write.js